# 01. 데이터 수집

## 목표
국내 은행 **연체율 예측 모델**에 필요한 두 종류의 데이터를 수집합니다.

| 데이터 | 출처 | 주기 |
|---|---|---|
| 거시경제 지표 (기준금리, 실업률, 대출금리 등) | 한국은행 ECOS API | 분기별 |
| 국내은행 원화대출 연체율 | 금융감독원 FISIS | 분기별 |

## API 키 준비 (필수)
실행 전에 프로젝트 루트에 `.env` 파일을 만들어야 합니다.

```
copy .env.example .env
```

그 후 `.env` 파일 안에 발급받은 키를 입력하세요.
- **ECOS 키 발급**: https://ecos.bok.or.kr → 회원가입 → 마이페이지 → OpenAPI 신청
- **공공데이터포털 키 발급**: https://www.data.go.kr → 회원가입 → 마이페이지 → 인증키 발급

In [ ]:
# ── 라이브러리 임포트 ──────────────────────────────────────────────────────────
import requests          # HTTP API 호출
import pandas as pd      # 데이터 처리
import os                # 환경변수 읽기
from pathlib import Path # 파일 경로 처리

# .env 파일에서 API 키 로드
try:
    from dotenv import load_dotenv
    # 노트북 위치(notebooks/)에서 상위 폴더로 이동해 .env를 찾음
    load_dotenv(Path('..') / '.env')
    print("✅ .env 파일 로드 성공")
except ImportError:
    print("⚠️  python-dotenv 미설치 → pip install python-dotenv")

# ── 경로 설정 ──────────────────────────────────────────────────────────────────
RAW_DIR = Path('..') / 'data' / 'raw'
PROCESSED_DIR = Path('..') / 'data' / 'processed'

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 원본 데이터 저장 위치: {RAW_DIR.resolve()}")
print(f"📁 가공 데이터 저장 위치: {PROCESSED_DIR.resolve()}")

---
## PART 1. 한국은행 ECOS API — 거시경제 지표 수집

### ECOS API 동작 방식
```
https://ecos.bok.or.kr/api/StatisticSearch/{API키}/json/kr/{시작}/9999/{통계코드}/{주기}/{시작일}/{종료일}/{항목코드}/
```

### 수집할 지표 (분기별, 2005Q1 ~ 2025Q4)
| 지표 | 통계코드 | 항목코드 | 단위 |
|---|---|---|---|
| 한국은행 기준금리 | 722Y001 | 0101000 | 연% |
| 예금금리 (정기예금 1년) | 121Y002 | BEABAA211 | 연% |
| 대출금리 (가계대출) | 121Y006 | BEABAB | 연% |
| 실업률 | 901Y027 | I61CA | % |
| 가계대출 잔액 | 151Y003 | BBKA | 십억원 |

In [ ]:
# ── ECOS API 공통 함수 ─────────────────────────────────────────────────────────

ECOS_KEY = os.getenv('ECOS_API_KEY', 'sample')
print(f"ECOS 키 상태: {'정식 키 사용 중 ✅' if ECOS_KEY != 'sample' else 'sample 키 사용 중 ⚠️'}")

# sample 키: 1회 호출당 최대 10건 제한 → 페이징 처리 필요
PAGE_SIZE = 10 if ECOS_KEY == 'sample' else 10000

def fetch_ecos(stat_code, cycle, start, end, item1='', item2='', item3='', item4=''):
    """
    ECOS API에서 통계 데이터를 페이징으로 전부 가져오는 함수.

    Parameters:
        stat_code : 통계표 코드 (예: '722Y001')
        cycle     : 주기 - 'A'(연), 'Q'(분기), 'M'(월)
        start     : 시작 기간 (예: '2005Q1', '200501')
        end       : 종료 기간 (예: '2025Q4', '202512')
        item1~4   : 항목 코드 (없으면 빈 문자열)

    Returns:
        pandas DataFrame
    """
    all_rows = []
    page_start = 1

    while True:
        page_end = page_start + PAGE_SIZE - 1
        url = (
            f"https://ecos.bok.or.kr/api/StatisticSearch/{ECOS_KEY}/json/kr"
            f"/{page_start}/{page_end}/{stat_code}/{cycle}/{start}/{end}"
            f"/{item1}/{item2}/{item3}/{item4}"
        )
        resp = requests.get(url, timeout=30)
        data = resp.json()

        if 'StatisticSearch' not in data:
            msg = data.get('RESULT', {}).get('MESSAGE', '알 수 없는 오류')
            if page_start == 1:
                print(f"  ⚠️  ECOS 오류 [{stat_code}]: {msg}")
            break

        rows = data['StatisticSearch']['row']
        all_rows.extend(rows)

        total = int(data['StatisticSearch']['list_total_count'])

        # 마지막 페이지면 종료
        if page_end >= total:
            break

        page_start += PAGE_SIZE

    return pd.DataFrame(all_rows)

print("ECOS 공통 함수 정의 완료 ✅ (페이징 자동 처리)")

In [ ]:
# ── 1-1. 한국은행 기준금리 수집 (분기별) ──────────────────────────────────────
print("[1/5] 기준금리 수집 중...")

df_base_rate = fetch_ecos(
    stat_code='722Y001',   # 한국은행 기준금리 및 여수신금리
    cycle='Q',             # 분기별
    start='2005Q1',
    end='2025Q4',
    item1='0101000'        # 한국은행 기준금리
)

if not df_base_rate.empty:
    # 필요한 컬럼만 선택하고 이름 변경
    df_base_rate = df_base_rate[['TIME', 'DATA_VALUE']].rename(
        columns={'TIME': 'quarter', 'DATA_VALUE': 'base_rate'}
    )
    df_base_rate['base_rate'] = pd.to_numeric(df_base_rate['base_rate'])
    print(f"  → {len(df_base_rate)}건 수집 완료 ({df_base_rate['quarter'].min()} ~ {df_base_rate['quarter'].max()})")
    df_base_rate.head()

In [ ]:
# ── 1-2. 예금금리 수집 (정기예금 1년, 분기별) ─────────────────────────────────
print("[2/5] 예금금리 수집 중...")

df_deposit_rate = fetch_ecos(
    stat_code='121Y002',   # 예금은행 수신금리(신규취급액 기준)
    cycle='Q',
    start='2005Q1',
    end='2025Q4',
    item1='BEABAA211'      # 정기예금
)

if not df_deposit_rate.empty:
    df_deposit_rate = df_deposit_rate[['TIME', 'DATA_VALUE']].rename(
        columns={'TIME': 'quarter', 'DATA_VALUE': 'deposit_rate'}
    )
    df_deposit_rate['deposit_rate'] = pd.to_numeric(df_deposit_rate['deposit_rate'])
    print(f"  → {len(df_deposit_rate)}건 수집 완료")
    df_deposit_rate.head()

In [ ]:
# ── 1-3. 대출금리 수집 (가계대출, 분기별) ────────────────────────────────────
print("[3/5] 대출금리(가계) 수집 중...")

df_loan_rate = fetch_ecos(
    stat_code='121Y006',   # 예금은행 여신금리(신규취급액 기준)
    cycle='Q',
    start='2005Q1',
    end='2025Q4',
    item1='BECBLA03'       # 가계대출
)

if not df_loan_rate.empty:
    df_loan_rate = df_loan_rate[['TIME', 'DATA_VALUE']].rename(
        columns={'TIME': 'quarter', 'DATA_VALUE': 'loan_rate'}
    )
    df_loan_rate['loan_rate'] = pd.to_numeric(df_loan_rate['loan_rate'])
    print(f"  → {len(df_loan_rate)}건 수집 완료")
    df_loan_rate.head()

In [ ]:
# ── 1-4. 실업률 수집 (분기별, 실업자/경제활동인구 → 계산) ─────────────────────
print("[4/5] 실업률 수집 중...")

# 실업자 수 (단위: 천명)
df_unemployed = fetch_ecos(
    stat_code='901Y027',
    cycle='Q',
    start='2005Q1',
    end='2025Q4',
    item1='I61BB',   # 실업자
    item2='I28A'     # 원계열
)

# 경제활동인구 (단위: 천명)
df_eap = fetch_ecos(
    stat_code='901Y027',
    cycle='Q',
    start='2005Q1',
    end='2025Q4',
    item1='I61B',    # 경제활동인구
    item2='I28A'
)

if not df_unemployed.empty and not df_eap.empty:
    df_unemployed = df_unemployed[['TIME', 'DATA_VALUE']].rename(
        columns={'TIME': 'quarter', 'DATA_VALUE': 'unemployed'}
    )
    df_eap = df_eap[['TIME', 'DATA_VALUE']].rename(
        columns={'TIME': 'quarter', 'DATA_VALUE': 'eap'}
    )

    df_unemployment = df_unemployed.merge(df_eap, on='quarter')
    df_unemployment['unemployed'] = pd.to_numeric(df_unemployment['unemployed'])
    df_unemployment['eap'] = pd.to_numeric(df_unemployment['eap'])

    # 실업률(%) = 실업자 / 경제활동인구 × 100
    df_unemployment['unemployment_rate'] = (
        df_unemployment['unemployed'] / df_unemployment['eap'] * 100
    ).round(2)

    df_unemployment = df_unemployment[['quarter', 'unemployment_rate']]
    print(f"  → {len(df_unemployment)}건 수집 완료")
    df_unemployment.head()

In [ ]:
# ── 1-5. 가계대출 잔액 수집 (분기별) ─────────────────────────────────────────
print("[5/5] 가계대출 잔액 수집 중...")

df_household_loan = fetch_ecos(
    stat_code='151Y001',   # 가계신용 통계
    cycle='Q',
    start='2005Q1',
    end='2025Q4',
    item1='1100000'        # 가계대출
)

if not df_household_loan.empty:
    df_household_loan = df_household_loan[['TIME', 'DATA_VALUE']].rename(
        columns={'TIME': 'quarter', 'DATA_VALUE': 'household_loan_balance'}
    )
    df_household_loan['household_loan_balance'] = pd.to_numeric(df_household_loan['household_loan_balance'])
    print(f"  → {len(df_household_loan)}건 수집 완료 (단위: 십억원)")
    df_household_loan.head()

In [ ]:
# ── 거시경제 지표 병합 및 저장 ─────────────────────────────────────────────────

frames = [
    df for df in [df_base_rate, df_deposit_rate, df_loan_rate, df_unemployment, df_household_loan]
    if 'df' in dir() and isinstance(df, __import__('pandas').DataFrame) and not df.empty
]

# 변수 존재 여부 확인 후 병합
available = []
for name, df in [
    ('기준금리',    df_base_rate       if 'df_base_rate'       in dir() else None),
    ('예금금리',    df_deposit_rate    if 'df_deposit_rate'    in dir() else None),
    ('대출금리',    df_loan_rate       if 'df_loan_rate'       in dir() else None),
    ('실업률',      df_unemployment    if 'df_unemployment'    in dir() else None),
    ('가계대출잔액', df_household_loan  if 'df_household_loan'  in dir() else None),
]:
    if df is not None and not df.empty:
        available.append(df)
        print(f"  ✅ {name}: {len(df)}건")
    else:
        print(f"  ⚠️  {name}: 없음")

if available:
    df_macro = available[0]
    for df in available[1:]:
        df_macro = df_macro.merge(df, on='quarter', how='outer')

    df_macro = df_macro.sort_values('quarter').reset_index(drop=True)

    save_path = RAW_DIR / 'macro_indicators.csv'
    df_macro.to_csv(save_path, index=False, encoding='utf-8-sig')
    print(f"\n✅ 거시경제 지표 저장: {save_path}")
    print(f"   → {len(df_macro)}행, {df_macro.shape[1]}열")
    df_macro.tail()

---
## PART 2. 연체율 데이터 수집

연체율은 **두 가지 방법** 중 하나로 수집합니다.

### 방법 A — 공공데이터포털 API (자동, 권장)
- `.env`에 `DATA_GO_KR_API_KEY` 설정 필요
- 데이터셋: 금융위원회_금융통계국내은행정보 (ID: 15061304)

### 방법 B — FISIS 수동 다운로드 (API 키 없을 때)
1. https://fisis.fss.or.kr 접속
2. **통계검색 → 경영공시 → 연체율** 메뉴 선택  
3. 기간: 2005년 ~ 2025년, 국내은행 전체 선택
4. Excel 다운로드 → 파일명을 `delinquency_rate_raw.xlsx`로 저장
5. `data/raw/` 폴더에 넣기

In [ ]:
# ── 방법 A: 공공데이터포털 API로 연체율 자동 수집 ──────────────────────────────

DATA_GO_KEY = os.getenv('DATA_GO_KR_API_KEY', '')

def fetch_delinquency_from_api(api_key, start_year=2005, end_year=2025):
    """
    공공데이터포털 API로 국내은행 연체율 데이터 수집.
    금융위원회_금융통계국내은행정보 API (ID: 15061304) 사용.
    """
    rows = []
    
    for year in range(start_year, end_year + 1):
        for quarter in range(1, 5):
            month = quarter * 3  # Q1→3월, Q2→6월, Q3→9월, Q4→12월
            params = {
                'serviceKey': api_key,
                'resultType' : 'json',
                'numOfRows'  : 100,
                'pageNo'     : 1,
                'basDt'      : f"{year}{month:02d}",  # 기준년월 (예: 202503)
            }
            url = "https://apis.data.go.kr/1160100/service/GetFinancialKbankInfoService/getKbankMainMgInfo"
            resp = requests.get(url, params=params, timeout=30)
            
            if resp.status_code != 200:
                continue
            
            data = resp.json()
            items = data.get('response', {}).get('body', {}).get('items', {}).get('item', [])
            
            if isinstance(items, dict):
                items = [items]
            
            for item in items:
                # 전체 국내은행 합산 행만 선택
                if item.get('fncCmpnyNm') in ['국내은행합계', '국내은행 합계', '합계']:
                    rows.append({
                        'quarter'           : f"{year}Q{quarter}",
                        'delinquency_total' : item.get('wncRatioAll'),    # 전체 연체율
                        'delinquency_corp'  : item.get('wncRatioCorp'),   # 기업대출 연체율
                        'delinquency_hh'    : item.get('wncRatioHh'),     # 가계대출 연체율
                    })
    
    return pd.DataFrame(rows)


if DATA_GO_KEY and DATA_GO_KEY != '여기에_공공데이터포털_키_입력':
    print("[방법 A] 공공데이터포털 API로 연체율 수집 중...")
    df_delinquency = fetch_delinquency_from_api(DATA_GO_KEY)
    
    if not df_delinquency.empty:
        save_path = RAW_DIR / 'delinquency_rate_api.csv'
        df_delinquency.to_csv(save_path, index=False, encoding='utf-8-sig')
        print(f"  ✅ API 수집 완료: {len(df_delinquency)}건 → {save_path}")
        df_delinquency.head()
    else:
        print("  ⚠️  API 응답 없음. 방법 B(수동 다운로드)를 사용하세요.")
else:
    print("[방법 A] 공공데이터포털 API 키 없음 → 방법 B(수동 다운로드)로 진행하세요.")
    print("  키 발급: https://www.data.go.kr → 회원가입 → 마이페이지 → 인증키 발급")

In [ ]:
# ── 방법 B: FISIS 수동 다운로드 파일 읽기 ─────────────────────────────────────
# API 키가 없거나 방법 A 실패 시 사용

manual_file = RAW_DIR / 'delinquency_rate_raw.xlsx'

if manual_file.exists():
    print("[방법 B] FISIS 수동 다운로드 파일 읽는 중...")
    
    # FISIS Excel 파일 구조에 맞게 조정 (첫 실행 후 구조 확인 필요)
    df_raw = pd.read_excel(manual_file, header=2)  # 헤더 위치는 파일에 따라 다를 수 있음
    
    print("파일 미리보기 (상위 5행):")
    print(df_raw.head())
    print(f"\n컬럼 목록: {df_raw.columns.tolist()}")
    print("\n⚠️  위 컬럼 확인 후 아래 코드의 컬럼명을 실제 파일에 맞게 수정하세요.")

else:
    print("[방법 B] 수동 다운로드 파일 없음")
    print(f"  저장 위치: {manual_file.resolve()}")
    print()
    print("  📌 다운로드 방법:")
    print("  1. https://fisis.fss.or.kr 접속")
    print("  2. 테마통계 → 연체율 → 국내 은행 원화대출 연체율")
    print("  3. 기간: 2005Q1 ~ 2025Q4, 전체 은행 선택")
    print("  4. 엑셀 다운로드 → delinquency_rate_raw.xlsx 로 저장")
    print(f"  5. data/raw/ 폴더에 넣기")

In [ ]:
# ── 방법 B 파싱: FISIS Excel → 표준 형식 변환 ─────────────────────────────────
# 이 셀은 방법 B로 파일을 다운받은 후 실행하세요.
# FISIS 파일 구조를 확인한 후 컬럼명을 수정하세요.

def parse_fisis_excel(file_path):
    """
    FISIS에서 다운로드한 연체율 Excel 파일을 표준 형식으로 변환.
    
    반환 형식:
        quarter (예: 2010Q1) | delinquency_total | delinquency_corp | delinquency_hh
    """
    df = pd.read_excel(file_path, header=2)
    
    # ── 아래 컬럼명은 실제 파일을 열어서 확인 후 수정하세요 ──
    # 예시 (실제 파일과 다를 수 있음):
    col_date   = df.columns[0]   # 날짜 컬럼
    col_total  = df.columns[1]   # 전체 연체율 컬럼
    col_corp   = df.columns[2]   # 기업대출 연체율 컬럼
    col_hh     = df.columns[3]   # 가계대출 연체율 컬럼
    
    df_clean = df[[col_date, col_total, col_corp, col_hh]].copy()
    df_clean.columns = ['quarter', 'delinquency_total', 'delinquency_corp', 'delinquency_hh']
    
    # 빈 행 제거
    df_clean = df_clean.dropna(subset=['quarter'])
    
    # 수치형 변환
    for col in ['delinquency_total', 'delinquency_corp', 'delinquency_hh']:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    
    return df_clean


if manual_file.exists():
    df_delinquency = parse_fisis_excel(manual_file)
    save_path = RAW_DIR / 'delinquency_rate_parsed.csv'
    df_delinquency.to_csv(save_path, index=False, encoding='utf-8-sig')
    print(f"✅ 연체율 파싱 완료: {len(df_delinquency)}건 저장")
    print(df_delinquency.head())
else:
    print("⚠️  delinquency_rate_raw.xlsx 파일 없음. 위 셀의 다운로드 안내를 따르세요.")

---
## PART 3. 데이터 통합

거시경제 지표와 연체율을 `quarter` 기준으로 합쳐서 최종 데이터셋을 만듭니다.

In [ ]:
# ── 최종 데이터셋 통합 ─────────────────────────────────────────────────────────

# 거시경제 지표와 연체율을 분기(quarter) 기준으로 병합
if 'df_macro' in dir() and 'df_delinquency' in dir():
    df_final = df_macro.merge(df_delinquency, on='quarter', how='inner')
    df_final = df_final.sort_values('quarter').reset_index(drop=True)
    
    print("\n=== 최종 데이터셋 요약 ===")
    print(f"기간: {df_final['quarter'].min()} ~ {df_final['quarter'].max()}")
    print(f"행 수: {len(df_final)}개 분기")
    print(f"열 수: {df_final.shape[1]}개 변수")
    print(f"\n결측값 현황:")
    print(df_final.isnull().sum())
    
    # processed/ 폴더에 저장
    save_path = PROCESSED_DIR / 'dataset.csv'
    df_final.to_csv(save_path, index=False, encoding='utf-8-sig')
    print(f"\n✅ 최종 데이터셋 저장: {save_path}")
    df_final.tail()
    
else:
    print("⚠️  거시경제 지표 또는 연체율 데이터가 없습니다.")
    print("   위 셀들을 순서대로 실행한 후 다시 시도하세요.")

In [ ]:
# ── 수집 결과 요약 ─────────────────────────────────────────────────────────────
print("=" * 50)
print("  데이터 수집 완료 요약")
print("=" * 50)

files = list(RAW_DIR.glob('*.csv')) + list(RAW_DIR.glob('*.xlsx'))
print(f"\n📁 data/raw/ 파일 목록:")
for f in files:
    size_kb = f.stat().st_size / 1024
    print(f"   - {f.name} ({size_kb:.1f} KB)")

files_p = list(PROCESSED_DIR.glob('*.csv'))
print(f"\n📁 data/processed/ 파일 목록:")
for f in files_p:
    size_kb = f.stat().st_size / 1024
    print(f"   - {f.name} ({size_kb:.1f} KB)")

print("\n다음 단계: 02_eda.ipynb (탐색적 데이터 분석)")